## UN demographic data provenance
All UN demographic files downloaded from the page: https://population.un.org/wpp/downloads?folder=Standard%20Projections&group=CSV%20format

### Population size
Second link "1950-2100, medium (gz)" in "Population row", with notes "Population on 01 January, by 5-year age groups.", URL as below.

In [ ]:
import pandas as pd
from datetime import datetime, UTC
from tb_macro.constants import DATA_PATH
from tb_macro.inputs import read_who_csv

url = "https://population.un.org/wpp/assets/Excel%20Files/1_Indicator%20(Standard)/CSV_FILES/WPP2024_Population1JanuaryByAge5GroupSex_Medium.csv.gz"
data = pd.read_csv(url)

select_columns = ["ISO3_code", "Location", "Time", "AgeGrp", "AgeGrpStart", "PopTotal"]
select_iso3s = ["KIR", "IDN", "VNM", "PHL", "BRA", "RUS", "IND", "CHN", "ZAF"]
select_data = data.loc[data["ISO3_code"].isin(select_iso3s), select_columns]
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
filename = f"un_population_{timestamp}.csv"
select_data.to_csv(DATA_PATH / "population" / filename)

### Mortality

Both links "1950-2100, medium (gz)" and "2024-2100, other scenarios (gz)", with notes "Population on 01 January, by 5-year age groups.", URL as below.

In [ ]:
url = "https://population.un.org/wpp/assets/Excel%20Files/1_Indicator%20(Standard)/CSV_FILES/WPP2024_DeathsBySingleAgeSex_Medium_1950-2023.csv.gz"
data_1 = pd.read_csv(url)
url = "https://population.un.org/wpp/assets/Excel%20Files/1_Indicator%20(Standard)/CSV_FILES/WPP2024_DeathsBySingleAgeSex_Medium_2024-2100.csv.gz"
data_2 = pd.read_csv(url)
data = pd.concat([data_1, data_2])
select_columns = ["ISO3_code", "Location", "Time", "AgeGrp", "AgeGrpStart", "DeathTotal"]
select_data = data.loc[data["ISO3_code"].isin(select_iso3s), select_columns]
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
filename = f"un_mortality_{timestamp}.csv"
select_data.to_csv(DATA_PATH / "population" / filename, index=False)

### Fertility
First link "1950-2100, single age (gz)", with notes "Age-specific Fertility Rate (ASFR), Percent Age-specific Fertility Rate (PASFR), Births (thousands)", URL as below.

In [ ]:
url = "https://population.un.org/wpp/assets/Excel%20Files/1_Indicator%20(Standard)/CSV_FILES/WPP2024_Fertility_by_Age1.csv.gz"
data = pd.read_csv(url)
select_columns = ["ISO3_code", "Location", "Time", "AgeGrp", "AgeGrpStart", "ASFR", "Variant"]
select_data = data.loc[data["ISO3_code"].isin(select_iso3s) & (data["Variant"] == "Medium"), select_columns]
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
filename = f"un_fertility_{timestamp}.csv"
select_data.to_csv(DATA_PATH / "population" / filename, index=False)

## WHO data

### Treatment outcomes
"Treatment outcomes [>1Mb]" link from https://www.who.int/teams/global-programme-on-tuberculosis-and-lung-health/data,
with data dictionary listed on the same page at https://extranet.who.int/tme/generateCSV.asp?ds=dictionary
and URL as below.

In [ ]:
data = read_who_csv("outcomes")
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
filename = f"who_outcomes_{timestamp}.csv"
data.to_csv(DATA_PATH / f"who/{filename}", index=False)


### Indicators
"WHO TB burden estimates [>1Mb]" link from https://www.who.int/teams/global-programme-on-tuberculosis-and-lung-health/data,
with same data dictionary as mentioned above and URL as below.

In [ ]:
data = read_who_csv("estimates")

select_data = data.loc[data["iso3"].isin(select_iso3s)]
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
filename = f"who_indicators_{timestamp}.csv"
select_data.to_csv(DATA_PATH / "who" / filename, index=False)


## World Bank data

### Indicators
WDI API: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392

- GDP per capita, constant 2015 USD (`NY.GDP.PCAP.KD`)
- Prevalence of undernourishment (`SN.ITK.DEFC.ZS`)

In [ ]:
import json
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

indicators = ["NY.GDP.PCAP.KD", "SN.ITK.DEFC.ZS"]  # GDP pc; prevalence of undernourishment
start_year, end_year = 1960, 2024

rows = []
for indicator in indicators:
    params = {
        "format": "json",
        "per_page": 20000,
        "date": f"{start_year}:{end_year}",
    }
    url = (
        f"https://api.worldbank.org/v2/country/{';'.join(select_iso3s)}"
        f"/indicator/{indicator}?{urlencode(params)}"
    )
    with urlopen(url) as resp:
        payload = json.load(resp)
    for item in payload[1]:
        rows.append(
            {
                "iso3": item["countryiso3code"],
                "country": item["country"]["value"],
                "indicator_id": item["indicator"]["id"],
                "indicator": item["indicator"]["value"],
                "year": int(item["date"]),
                "value": item["value"],
            }
        )

select_data = pd.DataFrame(rows).sort_values(["iso3", "year"])
out_dir = Path(DATA_PATH) / "worldbank"
out_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
filename = f"wdi_{timestamp}.csv"
select_data.to_csv(out_dir / filename, index=False)